# Data Cleaning

This notebook loads the raw stock data collected from Yahoo Finance,
cleans it, computes daily returns, and exports the cleaned dataset for
subsequent analysis.

In [9]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

In [10]:
df = pd.read_csv("../data/raw/stock_data.csv")

df.head()

,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
0,2015-01-02,14.620667,14.620667,14.883333,14.217333,14.858000,71466000,TSLA
1,2015-01-05,14.006000,14.006000,14.433333,13.810667,14.303333,80527500,TSLA
2,2015-01-06,14.085333,14.085333,14.280000,13.614000,14.004000,93928500,TSLA
3,2015-01-07,14.063333,14.063333,14.318667,13.985333,14.223333,44526000,TSLA
4,2015-01-08,14.041333,14.041333,14.253333,14.000667,14.187333,51637500,TSLA


In [11]:
print(df.info())

print()

print(df.shape)

print()

print(df.columns)

<class 'pandas.DataFrame'>
RangeIndex: 8664 entries, 0 to 8663
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       8664 non-null   str    
 1   Adj Close  8664 non-null   float64
 2   Close      8664 non-null   float64
 3   High       8664 non-null   float64
 4   Low        8664 non-null   float64
 5   Open       8664 non-null   float64
 6   Volume     8664 non-null   int64  
 7   Ticker     8664 non-null   str    
dtypes: float64(5), int64(1), str(2)
memory usage: 541.6 KB
None

(8664, 8)

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume',
       'Ticker'],
      dtype='str')


In [12]:
df.isnull().sum()

Date         0
Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
Ticker       0
dtype: int64

In [13]:
df = df.dropna().reset_index(drop=True)

In [14]:
df["Date"] = pd.to_datetime(df["Date"])

Convert Numeric Columns

In [15]:
numeric_cols = ["Open", "High", "Low", "Close", "Volume"]

# Include Adj Close only if it exists
if "Adj Close" in df.columns:
    numeric_cols.append("Adj Close")

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [16]:
df = df.sort_values(
    by=["Ticker", "Date"]
).reset_index(drop=True)

Calculate Daily Returns

In [17]:
df["Daily_Return"] = (
    df.groupby("Ticker")["Close"]
      .pct_change()
)

Remove First Row of Each Stock

In [18]:
df = df.dropna().reset_index(drop=True)

In [19]:
df[
    ["Ticker", "Date", "Close", "Daily_Return"]
].head(10)

,Ticker,Date,Close,Daily_Return
0,BND,2015-01-05,82.889999,0.002904
1,BND,2015-01-06,83.129997,0.002895
2,BND,2015-01-07,83.180000,0.000602
3,BND,2015-01-08,83.050003,-0.001563
4,BND,2015-01-09,83.190002,0.001686
5,BND,2015-01-12,83.300003,0.001322
6,BND,2015-01-13,83.379997,0.000960
7,BND,2015-01-14,83.570000,0.002279
8,BND,2015-01-15,83.940002,0.004427
9,BND,2015-01-16,83.690002,-0.002978


In [20]:
df.isnull().sum()

Date            0
Adj Close       0
Close           0
High            0
Low             0
Open            0
Volume          0
Ticker          0
Daily_Return    0
dtype: int64

Summary Statistics

In [21]:
df.describe()

,Date,Adj Close,Close,High,Low,Open,Volume,Daily_Return
count,8661,8661.000000,8661.000000,8661.000000,8661.000000,8661.000000,8.661000e+03,8661.000000
mean,2020-09-27 23:38:33.127814,188.964803,201.140829,202.901750,199.225768,201.127108,6.631926e+07,0.000759
min,2015-01-05 00:00:00,9.578000,9.578000,10.331333,9.403333,9.488000,0.000000e+00,-0.210628
25%,2017-11-13 00:00:00,62.675869,74.070000,74.209999,73.930000,74.059998,6.247600e+06,-0.003935
50%,2020-09-28 00:00:00,133.455338,133.455338,136.243332,126.370003,131.659332,6.164960e+07,0.000356
75%,2023-08-11 00:00:00,279.429993,294.350006,298.459991,291.420013,295.333344,9.453750e+07,0.005038
max,2026-06-29 00:00:00,757.618225,759.570007,760.400024,756.750000,758.150024,9.140820e+08,0.226900
std,NaN,169.820710,171.950000,173.247238,170.515178,171.938852,6.553273e+07,0.021870


Dataset Information

In [22]:
print("Rows:", df.shape[0])

print("Columns:", df.shape[1])

print()

print(df["Ticker"].value_counts())

Rows: 8661
Columns: 9

Ticker
BND     2887
SPY     2887
TSLA    2887
Name: count, dtype: int64


In [23]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [24]:
df.to_csv(
    "../data/processed/clean_stock_data.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.


In [25]:
clean = pd.read_csv(
    "../data/processed/clean_stock_data.csv"
)

clean.head()

,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,Daily_Return
0,2015-01-05,59.577908,82.889999,82.919998,82.699997,82.739998,5820100,BND,0.002904
1,2015-01-06,59.750416,83.129997,83.379997,83.029999,83.029999,3887600,BND,0.002895
2,2015-01-07,59.786331,83.180000,83.279999,83.050003,83.139999,2433400,BND,0.000602
3,2015-01-08,59.692890,83.050003,83.110001,82.970001,83.110001,1873400,BND,-0.001563
4,2015-01-09,59.793522,83.190002,83.290001,83.000000,83.010002,1646100,BND,0.001686


In [26]:
print(clean.info())

print()

print(clean.columns)

<class 'pandas.DataFrame'>
RangeIndex: 8661 entries, 0 to 8660
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          8661 non-null   str    
 1   Adj Close     8661 non-null   float64
 2   Close         8661 non-null   float64
 3   High          8661 non-null   float64
 4   Low           8661 non-null   float64
 5   Open          8661 non-null   float64
 6   Volume        8661 non-null   int64  
 7   Ticker        8661 non-null   str    
 8   Daily_Return  8661 non-null   float64
dtypes: float64(6), int64(1), str(2)
memory usage: 609.1 KB
None

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker',
       'Daily_Return'],
      dtype='str')


In [27]:
print(df.head())
print(df.columns)
print(df.shape)
print(df.isnull().sum())

        Date  Adj Close      Close       High        Low       Open   Volume  \
0 2015-01-05  59.577908  82.889999  82.919998  82.699997  82.739998  5820100   
1 2015-01-06  59.750416  83.129997  83.379997  83.029999  83.029999  3887600   
2 2015-01-07  59.786331  83.180000  83.279999  83.050003  83.139999  2433400   
3 2015-01-08  59.692890  83.050003  83.110001  82.970001  83.110001  1873400   
4 2015-01-09  59.793522  83.190002  83.290001  83.000000  83.010002  1646100   

  Ticker  Daily_Return  
0    BND      0.002904  
1    BND      0.002895  
2    BND      0.000602  
3    BND     -0.001563  
4    BND      0.001686  
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker',
       'Daily_Return'],
      dtype='str')
(8661, 9)
Date            0
Adj Close       0
Close           0
High            0
Low             0
Open            0
Volume          0
Ticker          0
Daily_Return    0
dtype: int64
